# 05 - End-to-End Scoring Chain

In [1]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from app.core.config import MODEL_PATH
from app.services.feature_pipeline import build_features
from app.services.model import assign_risk_tier, get_model
from app.services.scoring import load_customers, predict_proba

## Proving it on 5 customers first

In [2]:
# Pull 5 real customerIDs to score -- from the same reproducible test
# split used throughout Day 3/4, so these are genuinely held-out.
raw_full = pd.read_csv("../data/raw/churn_data.csv")
df_full = build_features(raw_full)
_, X_test, _, _ = train_test_split(
    df_full.drop(columns=["churn_flag"]), df_full["churn_flag"],
    test_size=0.2, stratify=df_full["churn_flag"], random_state=42,
)
five_ids = raw_full.loc[X_test.index, "customerID"].head(5).tolist()
five_ids

['4376-KFVRS', '2754-SDJRD', '9917-KWRBE', '0365-GXEZS', '9385-NXKDA']

In [3]:
# Stage 1: load_customers
customers = load_customers("../data/raw/churn_data.csv", customer_ids=five_ids)
print("Stage 1 (load_customers):", customers.shape)
customers[["customerID", "Contract", "tenure"]]

Stage 1 (load_customers): (5, 20)


,customerID,Contract,tenure
0,4376-KFVRS,Two year,72
1,9917-KWRBE,One year,41
2,2754-SDJRD,Month-to-month,8
3,9385-NXKDA,Two year,72
4,0365-GXEZS,Month-to-month,18


In [4]:
# Stage 2: build_features
features = build_features(customers)
print("Stage 2 (build_features):", features.shape)
features.head()

Stage 2 (build_features): (5, 20)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,contract_risk,num_services,charge_trend,is_electronic_check,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,72,114.05,8468.20,0,6,-3.563889,0,True,True,True,True,False,True,True,False,True,True,False,False
1,0,41,78.35,3211.20,1,4,0.028049,0,False,True,True,True,False,True,False,False,True,True,False,False
2,1,8,100.15,908.55,2,3,-13.418750,0,False,False,False,True,False,True,True,False,True,True,False,False
3,0,72,82.65,5919.35,0,5,0.436806,0,False,True,False,True,False,True,False,False,True,True,False,False
4,0,18,78.20,1468.75,2,2,-3.397222,1,True,True,False,True,False,False,True,False,False,False,True,False


In [5]:
# Stage 3: predict_proba
artifact = get_model(path=Path("..") / MODEL_PATH)
probabilities = predict_proba(features, artifact)
print("Stage 3 (predict_proba):", probabilities.shape)
probabilities

Stage 3 (predict_proba): (5,)


array([7.0927493e-02, 3.2563686e-01, 9.7638470e-01, 5.4241705e-04,
       7.2867721e-01], dtype=float32)

In [6]:
# Stage 4: assign_tier
tiers = [assign_risk_tier(p) for p in probabilities]
print("Stage 4 (assign_tier):", len(tiers))
tiers

Stage 4 (assign_tier): 5


['low', 'low', 'high', 'low', 'high']

In [7]:
# Output DataFrame
result_5 = pd.DataFrame({
    "customerID": customers["customerID"],
    "churn_probability": probabilities,
    "risk_tier": tiers,
})
result_5

,customerID,churn_probability,risk_tier
0,4376-KFVRS,0.070927,low
1,9917-KWRBE,0.325637,low
2,2754-SDJRD,0.976385,high
3,9385-NXKDA,0.000542,low
4,0365-GXEZS,0.728677,high


## Composing the full chain

In [8]:
def run_scoring_chain(path: str | Path, customer_ids: list[str] | None = None) -> pd.DataFrame:
    """load_customers -> build_features -> predict_proba -> assign_tier -> output DataFrame.

    Notebook-local orchestrator kept for the staged demonstration below;
    app/services/scoring.py's score_all_customers() is the production
    version of this same chain (it also adds last_scored_at).
    """
    customers = load_customers(path, customer_ids=customer_ids)
    features = build_features(customers)
    artifact = get_model(path=Path("..") / MODEL_PATH)
    probabilities = predict_proba(features, artifact)
    tiers = [assign_risk_tier(p) for p in probabilities]

    return pd.DataFrame(
        {
            "customerID": customers["customerID"],
            "churn_probability": probabilities,
            "risk_tier": tiers,
        }
    )


# Same 5 customers, through the composed chain -- confirms it reproduces
# the manually-staged result above exactly, not just structurally.
check = run_scoring_chain("../data/raw/churn_data.csv", customer_ids=five_ids)
print("Matches manual staged run:", check.equals(result_5))
check

Matches manual staged run: True


,customerID,churn_probability,risk_tier
0,4376-KFVRS,0.070927,low
1,9917-KWRBE,0.325637,low
2,2754-SDJRD,0.976385,high
3,9385-NXKDA,0.000542,low
4,0365-GXEZS,0.728677,high


## Scaling to 100 customers

In [9]:
hundred_ids = raw_full.loc[X_test.index, "customerID"].head(100).tolist()
result_100 = run_scoring_chain("../data/raw/churn_data.csv", customer_ids=hundred_ids)

print("Shape:", result_100.shape)
print("\nTier distribution:")
print(result_100["risk_tier"].value_counts())
result_100.sort_values("churn_probability", ascending=False).head(10)

Shape: (100, 3)

Tier distribution:
risk_tier
low       54
high      27
medium    19
Name: count, dtype: int64


,customerID,churn_probability,risk_tier
32,2754-SDJRD,0.976385,high
9,9851-KIELU,0.974726,high
96,4415-IJZTP,0.973102,high
88,0871-URUWO,0.963951,high
16,1768-ZAIFU,0.960018,high
39,9907-SWKKF,0.941546,high
6,0970-ETWGE,0.931603,high
36,4927-WWOOZ,0.925822,high
74,8821-XNHVZ,0.917440,high
29,7181-BQYBV,0.893902,high
